In [1]:
!pip install "accelerate>=1.1.0" "transformers[torch]" datasets

In [2]:
import pandas as pd
import torch
import os
from transformers import T5Tokenizer, T5ForConditionalGeneration, Seq2SeqTrainer, Seq2SeqTrainingArguments
from datasets import Dataset

# Assuming the notebook is unzipped in /kaggle/working/notebooks
data_path = "/kaggle/input/datasets/subhan1501/text-summarizatio/arxiv_sample.csv"

print("Loading dataset...")
df = pd.read_csv(data_path, nrows=75000) 
df = df.dropna(subset=['abstract', 'title'])
print(f"Loaded {len(df)} clean rows for T5 training.")

Loading dataset...
Loaded 75000 clean rows for T5 training.


In [3]:
# We use t5-base as it provides a great balance of speed and accuracy
model_name = "t5-base"
print(f"Downloading {model_name}...")

tokenizer = T5Tokenizer.from_pretrained(model_name, legacy=False)
model = T5ForConditionalGeneration.from_pretrained(model_name)

dataset = Dataset.from_pandas(df)

def tokenize_function(examples):
    # 💥 CRITICAL T5 REQUIREMENT: Add the task prefix
    inputs = ["summarize: " + text for text in examples["abstract"]]
    
    model_inputs = tokenizer(inputs, max_length=128, truncation=True, padding="max_length")
    labels = tokenizer(examples["title"], max_length=32, truncation=True, padding="max_length")
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

print("Tokenizing dataset for T5...")
tokenized_datasets = dataset.map(tokenize_function, batched=True)

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Tokenizing dataset for T5...


Map:   0%|          | 0/75000 [00:00<?, ? examples/s]

In [4]:
# Create the specific T5 output directory
save_path = "../models/Model_T5"
os.makedirs(f"{save_path}/checkpoints", exist_ok=True)

print(f"Hardware Check: Using {'GPU' if torch.cuda.is_available() else 'CPU'}")

args = Seq2SeqTrainingArguments(
    output_dir=f"{save_path}/checkpoints",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    weight_decay=0.01,
    save_strategy="epoch", # Safe disk storage
    save_total_limit=1,
    num_train_epochs=1,    # 1 epoch for quick Kaggle turnaround
    logging_steps=100,
    eval_strategy="no",
    fp16=True              # Mixed precision for T4 GPUs
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets,
    processing_class=tokenizer,
)

print("🔥 Starting T5 GPU Training Loop...")
trainer.train()

# Save the final fine-tuned model
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"✅ T5 Training complete! Model securely saved to {save_path}")

Hardware Check: Using GPU
🔥 Starting T5 GPU Training Loop...


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss
100,5.938459
200,3.142653
300,2.986667
400,2.918874
500,2.854330
600,2.858120
700,2.806503
800,2.815441
900,2.718867
1000,2.782389


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ T5 Training complete! Model securely saved to ../models/Model_T5


In [5]:
import shutil
import os
from IPython.display import FileLink

# The path where we saved the model in Cell 4
model_folder = "../models/Model_T5"
zip_filename = "BART_Finetuned_Model" # Do not add .zip here, shutil handles it

print("📦 Zipping the fine-tuned model (this may take a minute)...")
shutil.make_archive(zip_filename, 'zip', model_folder)
print("✅ Zip created successfully!")

print("Click the link below to download manually if the auto-download fails:")
display(FileLink(f"{zip_filename}.zip"))

📦 Zipping the fine-tuned model (this may take a minute)...
✅ Zip created successfully!
Click the link below to download manually if the auto-download fails:


/kaggle/working/BART_Finetuned_Model.zip

In [6]:
from IPython.display import HTML

# HTML and Javascript to automatically click the download link
auto_download_code = f"""
<a id="auto_download" href="{zip_filename}.zip" download="{zip_filename}.zip" style="display:none;"></a>
<script>
    setTimeout(function() {{
        document.getElementById('auto_download').click();
    }}, 1000); // 1 second delay to ensure the file is ready
</script>
"""

display(HTML(auto_download_code))
print("⬇️ Automatic download triggered!")
print("⚠️ Note: If nothing happens, check your browser's address bar—it might be blocking pop-ups/downloads from Kaggle.")

⬇️ Automatic download triggered!
⚠️ Note: If nothing happens, check your browser's address bar—it might be blocking pop-ups/downloads from Kaggle.
